# 03 - Inference & Submission
Prediksi test set (dengan TTA) lalu simpan `submission.csv`.

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/azizahadilaa/document-image-classification-kaggle.git" 
REPO_DIR = "/kaggle/working/document-image-classification-kaggle"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
sys.path.append(REPO_DIR)

import pandas as pd, numpy as np, torch
from scripts.utils import set_seed, find_data_dir
set_seed(42)
DATA_DIR = find_data_dir()
CACHE_DIR = "/kaggle/temp/cache"
IMG_SIZE = 224
train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")
print(DATA_DIR, train_df.shape, test_df.shape)

In [ ]:
from scripts.preprocess import cache_images
cache_images(test_df, DATA_DIR, CACHE_DIR, IMG_SIZE)
import glob
ckpts = glob.glob("/kaggle/working/checkpoint/*_best.pt") + glob.glob("/kaggle/input/**/*_best.pt", recursive=True)
print(ckpts); CKPT = ckpts[0]

In [ ]:
from scripts.model import build_model
from scripts.dataset import make_loader
from scripts.predict import predict_proba, make_submission
device = "cuda"
model = build_model("convnext_tiny", num_classes=train_df["label"].nunique(), pretrained=False)
model.load_state_dict(torch.load(CKPT, map_location="cpu"))
test_loader = make_loader(test_df, CACHE_DIR, IMG_SIZE, 128, train=False, is_test=True)
probs = predict_proba(model, test_loader, device, tta=True)
os.makedirs("/kaggle/working/submissions", exist_ok=True)
sub = make_submission(test_df["id"].tolist(), probs, f"{DATA_DIR}/sample_submission.csv", "/kaggle/working/submission.csv")
sub.to_csv("/kaggle/working/submissions/submission.csv", index=False)
print(sub.head(), sub.shape, sub["label"].value_counts().sort_index().tolist())